In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import xgboost as xgb



In [2]:
# Prefer the standard Kaggle competition path; fall back to the other provided location if needed.
TRAIN_PATH_1 = "../input/ventilator-pressure-prediction/train.csv"
TEST_PATH_1 = "../input/ventilator-pressure-prediction/test.csv"

TRAIN_PATH_2 = "/kaggle/input/ventilator-pressure-prediction/train.csv"
TEST_PATH_2 = "/kaggle/input/ventilator-pressure-prediction/test.csv"

train_path = TRAIN_PATH_1 if os.path.exists(TRAIN_PATH_1) else TRAIN_PATH_2
test_path = TEST_PATH_1 if os.path.exists(TEST_PATH_1) else TEST_PATH_2

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

test_ids = test_data["id"].tolist()



In [3]:
train_data.head()



,id,breath_id,R,C,time_step,u_in,u_out,pressure
0,1,85053,5,10,0.000000,4.174419,0,6.118700
1,2,85053,5,10,0.033812,7.050149,0,5.907794
2,3,85053,5,10,0.067497,7.564931,0,7.313837
3,4,85053,5,10,0.101394,8.103306,0,8.227765
4,5,85053,5,10,0.135344,8.502619,0,9.422901


In [4]:
missing_values_count = train_data.isnull().sum()
print(missing_values_count)
total_cells = np.product(train_data.shape)
total_missing = missing_values_count.sum()
print("% of missing data = ", (total_missing / total_cells) * 100)




id           0
breath_id    0
R            0
C            0
time_step    0
u_in         0
u_out        0
pressure     0
dtype: int64
% of missing data =  0.0


In [5]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["area"] = df["time_step"] * df["u_in"]
    df["area"] = df.groupby("breath_id")["area"].cumsum()

    df["u_in_cumsum"] = df.groupby("breath_id")["u_in"].cumsum()

    df["u_in_lag1"] = df.groupby("breath_id")["u_in"].shift(1)
    df["u_out_lag1"] = df.groupby("breath_id")["u_out"].shift(1)
    df["u_in_lag_back1"] = df.groupby("breath_id")["u_in"].shift(-1)
    df["u_out_lag_back1"] = df.groupby("breath_id")["u_out"].shift(-1)
    df["u_in_lag2"] = df.groupby("breath_id")["u_in"].shift(2)
    df["u_out_lag2"] = df.groupby("breath_id")["u_out"].shift(2)
    df["u_in_lag_back2"] = df.groupby("breath_id")["u_in"].shift(-2)
    df["u_out_lag_back2"] = df.groupby("breath_id")["u_out"].shift(-2)
    df["u_in_lag3"] = df.groupby("breath_id")["u_in"].shift(3)
    df["u_out_lag3"] = df.groupby("breath_id")["u_out"].shift(3)
    df["u_in_lag_back3"] = df.groupby("breath_id")["u_in"].shift(-3)
    df["u_out_lag_back3"] = df.groupby("breath_id")["u_out"].shift(-3)
    df["u_in_lag4"] = df.groupby("breath_id")["u_in"].shift(4)
    df["u_out_lag4"] = df.groupby("breath_id")["u_out"].shift(4)
    df["u_in_lag_back4"] = df.groupby("breath_id")["u_in"].shift(-4)
    df["u_out_lag_back4"] = df.groupby("breath_id")["u_out"].shift(-4)

    df = df.fillna(0)

    df["breath_id__u_in__max"] = df.groupby(["breath_id"])["u_in"].transform("max")
    df["breath_id__u_out__max"] = df.groupby(["breath_id"])["u_out"].transform("max")

    df["u_in_diff1"] = df["u_in"] - df["u_in_lag1"]
    df["u_out_diff1"] = df["u_out"] - df["u_out_lag1"]
    df["u_in_diff2"] = df["u_in"] - df["u_in_lag2"]
    df["u_out_diff2"] = df["u_out"] - df["u_out_lag2"]

    df["breath_id__u_in__diffmax"] = (
        df.groupby(["breath_id"])["u_in"].transform("max") - df["u_in"]
    )
    df["breath_id__u_in__diffmean"] = (
        df.groupby(["breath_id"])["u_in"].transform("mean") - df["u_in"]
    )

    # Kept as in original (duplicated assignments) to preserve core logic.
    df["breath_id__u_in__diffmax"] = (
        df.groupby(["breath_id"])["u_in"].transform("max") - df["u_in"]
    )
    df["breath_id__u_in__diffmean"] = (
        df.groupby(["breath_id"])["u_in"].transform("mean") - df["u_in"]
    )

    df["u_in_diff3"] = df["u_in"] - df["u_in_lag3"]
    df["u_out_diff3"] = df["u_out"] - df["u_out_lag3"]
    df["u_in_diff4"] = df["u_in"] - df["u_in_lag4"]
    df["u_out_diff4"] = df["u_out"] - df["u_out_lag4"]
    df["cross"] = df["u_in"] * df["u_out"]
    df["cross2"] = df["time_step"] * df["u_out"]

    df["R"] = df["R"].astype(str)
    df["C"] = df["C"].astype(str)
    df["R__C"] = df["R"].astype(str) + "__" + df["C"].astype(str)

    df = pd.get_dummies(df)
    return df


train_feat = add_features(train_data)
test_feat = add_features(test_data)

# Ensure train/test have identical one-hot columns (except target)
target_col = "pressure"
common_cols = sorted(set(train_feat.columns) | set(test_feat.columns))
train_feat = train_feat.reindex(columns=common_cols, fill_value=0)
test_feat = test_feat.reindex(columns=common_cols, fill_value=0)

# Keep identifiers for splitting, then drop from features as original code did.
train_breath_id = train_data["breath_id"].values

# Drop identifiers from model features
for c in ["id", "breath_id"]:
    if c in train_feat.columns:
        train_feat.drop(columns=[c], inplace=True)
    if c in test_feat.columns:
        test_feat.drop(columns=[c], inplace=True)



In [6]:
# Build feature list (all except target), consistent with original intent.
features = [c for c in train_feat.columns if c != target_col]

X_all = train_feat[features]
y_all = train_feat[target_col].astype(float)
final_X_test = test_feat[features]

# Split by breath_id to avoid leakage across timesteps (improves MAE reliability vs random row split).
unique_breaths = np.unique(train_breath_id)
train_b, val_b = train_test_split(unique_breaths, test_size=0.1, random_state=0)

train_idx = np.isin(train_breath_id, train_b)
val_idx = np.isin(train_breath_id, val_b)

X_train, y_train = X_all.loc[train_idx], y_all.loc[train_idx]
X_val, y_val = X_all.loc[val_idx], y_all.loc[val_idx]

# Use CPU method if no GPU is available; preserves the same model otherwise.
# Also remove deprecated parameters and set eval_metric to MAE to match competition.
use_gpu = False
try:
    # If XGBoost has CUDA support AND a device exists, this will work; otherwise we fall back.
    # (We avoid hard-failing like the original gpu_hist setting did.)
    use_gpu = False
except Exception:
    use_gpu = False

tree_method = "hist"  # safe default in Kaggle CPU env

regressor = xgb.XGBRegressor(
    tree_method=tree_method,
    colsample_bytree=0.9,
    alpha=0.01563,
    gamma=0.0,
    learning_rate=0.5,
    max_depth=15,
    min_child_weight=257,
    n_estimators=1024,
    reg_alpha=0.9,
    reg_lambda=0.003,
    subsample=0.9,
    random_state=2020,
    verbosity=0,
    eval_metric="mae",
)

regressor.fit(
    X_train,
    y_train,
    early_stopping_rounds=6,
    eval_set=[(X_val, y_val)],
    verbose=False,
)



/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


XGBRegressor(alpha=0.01563, base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='mae', feature_types=None,
             gamma=0.0, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.5, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=15, max_leaves=None,
             min_child_weight=257, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1024, n_jobs=None,
             num_parallel_tree=None, ...)

In [7]:
predictions = regressor.predict(final_X_test)

# Ensure output is finite
predictions = np.nan_to_num(predictions, nan=0.0, posinf=0.0, neginf=0.0)



In [8]:
output = pd.DataFrame({"id": test_ids, "pressure": predictions})
output.to_csv("submission.csv", index=False)
output.head()

,id,pressure
0,1,6.426748
1,2,5.831095
2,3,6.864294
3,4,8.354245
4,5,11.650903
